# 76 — Bug #1 probe: is `history_tids` empty at serve?

Confirms the root-cause finding that the played-track list the session channels depend on is empty in production, because of a role-string mismatch.

Mechanism: the dev/blind inference parsers rewrite every music turn to `role == "assistant"` (run_inference_devset.py:40-44, run_inference_blindset.py:31-41), but `crs_baseline.batch_chat` builds `history_tids` by filtering `role == "music"` (crs_baseline.py:603-606) — so it never matches and `history_tids == []` at serve.

Downstream of empty `history_tids`:
- SASRec loses its played-track sequence conditioning (sasrec_seq.py:60, reads `history_tids` directly, no fallback).
- The LGBM session-continuity features (same_artist / same_album / tag_overlap) collapse to 0 (crs_baseline.py:650-651).
- same_artist / session_cf channels use `played_tids_from_context`, which has a track_id fallback — so they survive at BLIND-serve (parser carries track_id) but die at DEV-serve (no track_id).

No GPU, no models. Pure parser + filter logic copied VERBATIM from the repo (citations inline), run on the real dev dataset.

HOW TO RUN: run cell 1 (setup), then cell 2 (probe). ~1 minute total.

EXPECTED if the bug is real:
- CURRENT (role==music): 0 / N for BOTH parsers — this is the shipped behaviour.
- ROLE-FIX (role in music/assistant + track_id): N / N for BLIND, 0 / N for DEV (dev parser carries no track_id).
- FULL-FIX (role-fix + dev parser also attaches track_id): N / N for DEV.

In [ ]:
# 1) Setup — minimal. No GPU, no model libraries.
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # dataset access
except Exception:
    pass  # running locally; assumes HF_TOKEN already in env if the dataset is gated
!pip install -q "datasets" "pandas<3.0"
print('setup done')

In [ ]:
# 2) PROBE. All logic below is copied VERBATIM from the repo (citations inline)
#    so this cell is self-contained and faithful to the serve path.
import pandas as pd, collections
from datasets import load_dataset

# id_to_metadata returns expanded TEXT in production (NOT a catalog id), which is
# all that matters to the filter; stub it so we need no catalog download.
def _id_to_metadata(tid):
    return f'track metadata text for {tid}'

# --- VERBATIM: run_inference_devset.py chat_history_parser (music -> assistant,
#     content -> expanded TEXT, NO track_id attached). ---
def dev_parser(conversations, target_turn_number):
    df = pd.DataFrame(conversations)
    dfh = df[df['turn_number'] < target_turn_number]
    ch = []
    for t in dfh.to_dict('records'):
        role, content = t['role'], t['content']
        if t['role'] == 'music':                      # run_inference_devset.py:40-41
            role = 'assistant'
            content = _id_to_metadata(t['content'])
        ch.append({'role': role, 'content': content})
    return ch

# --- VERBATIM: run_inference_blindset.py parser (music -> assistant, content ->
#     expanded TEXT, BUT raw track_id ALSO carried on the turn). ---
def blind_parser(conversations, target_turn_number):
    df = pd.DataFrame(conversations)
    dfh = df[df['turn_number'] < target_turn_number]
    ch = []
    for t in dfh.to_dict('records'):
        role, content = t['role'], t['content']
        if t['role'] == 'music':                      # run_inference_blindset.py:31-41
            raw = t['content']
            ch.append({'role': 'assistant', 'content': _id_to_metadata(raw), 'track_id': raw})
            continue
        ch.append({'role': role, 'content': content})
    return ch

# --- FULL-FIX dev parser: same as dev_parser but ALSO attaches track_id ---
def dev_parser_FIXED(conversations, target_turn_number):
    df = pd.DataFrame(conversations)
    dfh = df[df['turn_number'] < target_turn_number]
    ch = []
    for t in dfh.to_dict('records'):
        role, content = t['role'], t['content']
        if t['role'] == 'music':
            raw = t['content']
            ch.append({'role': 'assistant', 'content': _id_to_metadata(raw), 'track_id': raw})
            continue
        ch.append({'role': role, 'content': content})
    return ch

# --- VERBATIM: crs_baseline.py:603-606 history_tids extraction (CURRENT, shipped) ---
def history_tids_CURRENT(prior_history):
    return [str(t.get('track_id') or t.get('content')) for t in prior_history
            if t.get('role') == 'music' and (t.get('track_id') or t.get('content'))]

# --- PROPOSED ROLE-FIX: accept assistant-rewritten music turns, require a real
#     track_id (mirrors session_history.played_tids_from_context's role set). ---
def history_tids_ROLEFIX(prior_history):
    return [str(t['track_id']) for t in prior_history
            if t.get('role') in ('music', 'assistant') and t.get('track_id')]

# --- VERBATIM: session_history.played_tids_from_context (what same_artist /
#     session_cf actually call). ---
def played_tids_from_context(ctx, catalog_tids):
    if not ctx:
        return []
    explicit = ctx.get('history_tids')
    if explicit:
        return [str(t) for t in explicit if str(t) in catalog_tids]
    out = []
    for turn in ctx.get('chat_history', []) or []:
        if turn.get('role') in ('music', 'assistant'):
            tid = turn.get('track_id')
            if tid is not None and str(tid) in catalog_tids:
                out.append(str(tid)); continue
            c = str(turn.get('content', ''))
            if c in catalog_tids:
                out.append(c)
    return out

# ---- run on the real dev split ----
db = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
N = 50
TURN = 5  # predict turn 5 -> history = turns 1..4 (usually contains played music)

catalog = set()
for item in db.select(range(N)):
    for t in item['conversations']:
        if t['role'] == 'music':
            catalog.add(str(t['content']))

st = collections.Counter()
for item in db.select(range(N)):
    convos = item['conversations']
    if sum(1 for t in convos if t['role'] == 'music' and t['turn_number'] < TURN) == 0:
        continue  # no played history before the target turn
    st['n'] += 1
    # CURRENT shipped behaviour (both parsers)
    st['DEV_cur']   += 1 if history_tids_CURRENT(dev_parser(convos, TURN)) else 0
    st['BLIND_cur'] += 1 if history_tids_CURRENT(blind_parser(convos, TURN)) else 0
    # ROLE-FIX only (relax the filter; dev still has no track_id)
    st['DEV_rolefix']   += 1 if history_tids_ROLEFIX(dev_parser(convos, TURN)) else 0
    st['BLIND_rolefix'] += 1 if history_tids_ROLEFIX(blind_parser(convos, TURN)) else 0
    # FULL-FIX (role-fix + dev parser attaches track_id)
    st['DEV_fullfix'] += 1 if history_tids_ROLEFIX(dev_parser_FIXED(convos, TURN)) else 0
    # what same_artist / session_cf actually receive at serve (CURRENT history_tids)
    st['DEV_pf']   += 1 if played_tids_from_context({'chat_history': dev_parser(convos, TURN),   'history_tids': []}, catalog) else 0
    st['BLIND_pf'] += 1 if played_tids_from_context({'chat_history': blind_parser(convos, TURN), 'history_tids': []}, catalog) else 0

n = st['n']
print(f'Sessions with real played history before turn {TURN}: {n}\n')
print('How many sessions get a NON-EMPTY played list (out of {}):\n'.format(n))
print('  history_tids  (feeds SASRec sequence + LGBM session features)')
print(f'    DEV   parser  CURRENT(role==music)      : {st["DEV_cur"]:3d}/{n}')
print(f'    BLIND parser  CURRENT(role==music)      : {st["BLIND_cur"]:3d}/{n}')
print(f'    DEV   parser  ROLE-FIX                  : {st["DEV_rolefix"]:3d}/{n}  (dev carries no track_id -> still 0)')
print(f'    BLIND parser  ROLE-FIX                  : {st["BLIND_rolefix"]:3d}/{n}  (track_id present -> recovered)')
print(f'    DEV   parser  FULL-FIX (+track_id)      : {st["DEV_fullfix"]:3d}/{n}')
print()
print('  played_tids_from_context  (feeds same_artist + session_cf channels)')
print(f'    DEV   parser  (CURRENT serve)           : {st["DEV_pf"]:3d}/{n}  (no track_id -> dead at dev-serve)')
print(f'    BLIND parser  (CURRENT serve)           : {st["BLIND_pf"]:3d}/{n}  (track_id fallback -> alive at blind-serve)')
print()
print('VERDICT:')
print(f'  bug #1 confirmed if history_tids CURRENT == 0/{n} for BOTH parsers (shipped behaviour).')
print(f'  role-fix recovers BLIND ({n}/{n}); DEV also needs the parser to attach track_id (FULL-FIX).')
print(f'  same_artist/session_cf already survive at blind-serve but are dead in the dev harness.')